In [1]:
import os
import json
import pandas as pd
import traceback

In [2]:
from langchain.chat_models import ChatOpenAI




ModuleNotFoundError: No module named 'langchain_community'

In [ ]:
from dotenv import load_dotenv
load_dotenv() # take environment variables from .env.

In [ ]:
KEY=os.getenv("OPENAI_API_KEY")

In [ ]:
llm=ChatOpenAI(openai_api_key=KEY,model="gpt-4-0613", temperature=0.5, max_tokens=2048)



In [ ]:
from langchain.llms import OpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.chains import SequentialChain
from langchain.callbacks import get_openai_callback
import pyPDF2




In [ ]:
RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here,"
            "b": "choice here,"
            "c": "choice here,"
            "d": "choice here,"
        }
        "correct": "correct answer"
    },
    "2": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here,"
            "b": "choice here,"
            "c": "choice here,"
            "d": "choice here,"
        }
        "correct": "correct answer"
    },
    "3": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here,"
            "b": "choice here,"
            "c": "choice here,"
            "d": "choice here,"
        }
        "correct": "correct answer"
    },
}

In [ ]:
TEMPLATE= """
Text:{text}
You are an expert MCQ maker. Given the above text, it is your job to \
create a quiz of {number} multiple choice questions for {subject} students in {tone} tone.
Make sure the questions are not repeated and check all the questions to be confirming the text as well.
Ensure to make {number} MCQs
### RESPONSE_JSON
{response_json}

"""

In [ ]:
 quiz_generation_template = PromptTemplate(
  input_variables=["text", "number", "subject", "tone", "response_json"],
  template=Template
)

quiz_chain=LLmchain(llm=llm, prompt=quiz_generation_prompt, output_key="quiz", verbose=True)


In [ ]:
TEMPLATE2="""
You are an expert english Grammarian and writer. Given a Multiple Choice Quiz for {student} students.\
You need to evaluate the complexity of the question and give a complete analysis of the quiz. Only use at max 50 words for complexity \
if the quiz is not at per with the cognitive and analytical abilities of the students.\
update the quiz questions which needs to be changed and change the tone such that it perfectly fits the students.
Quiz_MCQs
{quiz}

Check from an export English Writer of the above quiz:
"""

In [ ]:
quiz_evaluation_prompt=PromptTemplate(input_variables=["subject", "quiz"], template=TEMPLATE)

review_chain=LLMchain(llm=llm, prompt=quiz_evaluation_prompt, output_key="review", verbose=True)

In [ ]:
 generate_evaluate_chain=SequentialChain(chains=[quiz_chain, review-chain], input_variables=["text", "number", "subject", "tone", "response_json"],
                                        output_variables=["quiz", "review"], verbose=True,)

In [ ]:
file_path=r"C:\Users"

In [ ]:
with open(filepath, 'r') as file:
     TEXT = file.read()

In [ ]:
print(TEXT)

In [ ]:
# Serialize the Python dictionary into a JSON-formated string
json.dumps(RESPONSE_JSON)

In [ ]:
NUMBER=5
SUBJECT="machine learning"
TONE="simple"

In [ ]:
#https://puthon.langchain.com/docs/modules/model_io/llms/token_usage_tracking

# How to Setup Token Usage Tracking in Langchain
with get_openai_callback() as cb:
    response=generate_evaluate_chain(
        {
            "text": TEXT,
            "number": NUMBER,
            "subject": SUBJECT,
            "tone": TONE,
            "reponse_json": json.dumps(RESPONSE_JSON)
        }
    )

In [ ]:
print(f"Total Tokens:{cb.total_tokens}")
print(f"Prompt Tokens:{cb.prompt_tokens}")
print(f"Completion Tokens:{cb.completion_tokens}")
print(f"Total Cost:{cb.total_cost}")

In [ ]:
quiz=response.get("quiz")

In [ ]:
json.load(quiz)

In [ ]:
quiz_table_data = []
for key, value in quiz.items():
    mcq = value["mcq"]
    options = " | ".join(
        [
            f"{option}: {option_value}"
            for option, option_value in value["options"].items()
           ]
        )
    correct = value["correct"]
    quiz_table_data.append({"MCQ": mcq, "Choices": options, "Correct": correct}) 

In [ ]:
quiz_table_data

In [ ]:
quiz=pd.DataFrame(quiz_table_data)

In [ ]:
 quiz.to_csv("machinelearning.csv", index=False)